# Recipe Extraction Notebook

In [1]:
import json

with open("./parsed_recipes.json", "r") as f:
    parsed_recipes = json.load(f)

In [2]:
import sys
sys.path.append("..")

from food_data_extraction import FoodEmbedding

In [3]:
food_embedding = FoodEmbedding()

c:\Users\micha\Documents\School\University\3rd Year\Final Year Project\Code\recipe_extraction\..\food_data_extraction\FoodEmbedding.py:34: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  self.food_nutrient = pd.read_csv(


In [4]:
food_embedding.load(index_path = "../food_data_extraction/food_embedding.faiss")

Some nutrients seem to have multiple IDs, such as sugar having both 1063 and 2000. Therefore, arrays of the nutrient IDs are declared, starting with the preferred and most common IDs first, checking for all IDs before giving up and returning None.

In [5]:
NUTRIENT_ID_MAP = {
    "calories":       [1008,  # Energy (KCAL) — most common
                       2047,  # Energy, Atwater General Factors
                       2048], # Energy, Atwater Specific Factors
    "carbohydrates":  [1005,  # Carbohydrate, by difference
                       2039,  # Carbohydrates
                       1050], # Carbohydrate, by summation
    "sugar":          [1063,  # Sugars, Total
                       2000], # Total Sugars
    "protein":        [1003,  # Protein
                       1053], # Adjusted Protein
    "fat":            [1004,  # Total lipid (fat)
                       1085], # Total fat (NLEA)
    "saturated_fat":  [1258,  # Fatty acids, total saturated
                       1326], # Fatty acids, total saturated (NLEA)
    "fiber":          [1079,  # Fiber, total dietary
                       2033], # Total dietary fiber (AOAC 2011.25)
    "sodium":         [1093], # Sodium, Na
}


In [6]:
from models import NutritionalInformation

def get_nutritional_information(fdc_id: int | None) -> NutritionalInformation:
    """
    Given FDC food information, this returns the nutritional information for that ingredient based on the FDC data (per 100g)

    :param fdc_id: the FDC food ID for the ingredient
    :type fdc_id: int | None

    :returns: the nutritional information for the ingredient
    :rtype: NutritionalInformation
    """

    if fdc_id is None:
        return NutritionalInformation()
    
    fdc_nutritional_information = food_embedding.get_nutritional_information(fdc_id)

    nutrients_per_100g = {}

    for nutrient_name, nutrient_ids in NUTRIENT_ID_MAP.items():
        value = None

        for nutrient_id in nutrient_ids:
            nutrient = fdc_nutritional_information[fdc_nutritional_information["nutrient_id"] == nutrient_id]

            if not nutrient.empty:
                value = nutrient.iloc[0]["amount"]
                break

        nutrients_per_100g[nutrient_name] = value

    return NutritionalInformation(
        calories = nutrients_per_100g ["calories"],
        carbohydrates = nutrients_per_100g ["carbohydrates"],
        sugar = nutrients_per_100g ["sugar"],
        protein = nutrients_per_100g ["protein"],
        fat = nutrients_per_100g ["fat"],
        saturated_fat = nutrients_per_100g ["saturated_fat"],
        fiber = nutrients_per_100g ["fiber"],
        sodium = nutrients_per_100g ["sodium"],
        # the dietary tags are handled in a separate notebook
        is_gluten_free = None,
        is_lactose_free = None,
        is_vegetarian = None,
        is_vegan = None 
    )

In [7]:
def get_ingredient_weight(fdc_id: int | None, ingredient_weight: float | None) -> float | None:
    """
    Given an ingredient's FDC food result and weight, this returns the weight of that ingredient in grams. This weight is either the weight provided in the recipe, or if that is not available, the default portion size for that ingredient in the FDC data. If neither of those are available, then None is returned
    :param fdc_id: the FDC food ID for the ingredient
    :type fdc_id: int | None
    :param ingredient_weight: the weight of the ingredient in grams, or None if the weight is not known
    :type ingredient_weight: float | None

    :returns: the weight of the ingredient in grams, or None if it cannot be determined
    :rtype: float | None
    """

    if ingredient_weight is not None:
        return ingredient_weight
    
    if fdc_id is None:
        return None

    portion_size = food_embedding.get_portion_size(fdc_id)
    
    if portion_size.empty:
        return 100
    
    return portion_size ["gram_weight"].iloc [0]

In [8]:
from models import DietaryTag

def get_recipe_dietary_tags(recipe_categories: list[str]) -> list[DietaryTag]:
    """
    Obtains the dietary tags for a recipe based on its categories

    :param recipe_categories: the categories of the recipe, which may include dietary information such as "Wheat/Gluten-Free", "Dairy-Free", "Vegetarian", and "Vegan"
    :type recipe_categories: list[str]

    :returns: the dietary tags for the recipe
    :rtype: list[DietaryTag]
    """

    dietary_tags: list[DietaryTag] = []

    for category in recipe_categories:
        if category == "Wheat/Gluten-Free":
            dietary_tags.append(DietaryTag.GLUTEN_FREE)
        elif category == "Dairy-Free":
            dietary_tags.append(DietaryTag.LACTOSE_FREE)
        elif category == "Vegetarian":
            dietary_tags.append(DietaryTag.VEGETARIAN)
        elif category == "Vegan":
            dietary_tags.append(DietaryTag.VEGAN)

    return dietary_tags

In [9]:
from tqdm.auto import tqdm
from models import Ingredient, Recipe

unique_ingredients: dict[str, Ingredient] = {}
unique_ingredient_fdc_ids: dict[str, int | None] = {}
structured_recipes = []

for recipe in tqdm(parsed_recipes, desc = "Extracting Recipes"):
    structured_ingredients = []
    ingredient_amounts = []

    for ingredient_name, ingredient_weight in zip(recipe ["ingredient_names"], recipe ["ingredient_weights"]):
        if ingredient_name not in unique_ingredients:
            search_result = food_embedding.search(ingredient_name)
            fdc_id = search_result.iloc [0].fdc_id if not search_result.empty else None

            nutritional_information = get_nutritional_information(fdc_id)
            structured_ingredient = Ingredient(
                name = ingredient_name,
                nutritional_information = nutritional_information
            )

            unique_ingredients [ingredient_name] = structured_ingredient
            unique_ingredient_fdc_ids [ingredient_name] = fdc_id
        else:
            structured_ingredient = unique_ingredients [ingredient_name]
            fdc_id = unique_ingredient_fdc_ids [ingredient_name]

        ingredient_amount = get_ingredient_weight(fdc_id, ingredient_weight)

        structured_ingredients.append(structured_ingredient)
        ingredient_amounts.append(ingredient_amount)

    ingredient_amount_dict = dict(zip(structured_ingredients, ingredient_amounts))

    total_calories = sum(filter(None, [ingredient.nutritional_information.calories for ingredient in ingredient_amount_dict.keys()]))
    num_serves = total_calories // recipe ["calories"] if recipe ["calories"] else 1

    structured_recipe = Recipe(
        name = recipe ["title"],
        ingredients = ingredient_amount_dict,
        dietary_tags = get_recipe_dietary_tags(recipe ["categories"]),
        instructions = recipe ["directions"],
        serves = num_serves
    )
    structured_recipes.append(structured_recipe)

Extracting Recipes:   0%|          | 0/19836 [00:00<?, ?it/s]

In [10]:
structured_recipes_json = [recipe.to_dict() for recipe in structured_recipes]

In [11]:
json.dump(structured_recipes_json, open("./structured_recipes.json", "w"), indent = 4)

In [12]:
structured_ingredients_json = [ingredient.to_dict() for ingredient in unique_ingredients.values()]

In [13]:
json.dump(structured_ingredients_json, open("./structured_ingredients.json", "w"), indent = 4)